## Implementação inicial do Thompson Sampling

Nesta etapa, primeiro foi implementado uma primeira versão do **Thompson Sampling clássico**, utilizando uma abordagem **Beta-Bernoulli**. O objetivo desta implementação é compreender o funcionamento do algoritmo antes de incorporar as características dos clientes no modelo contextual.

Em um segundo momento, foi incorporado o contexto do cliente, utilizando suas características para estimar diferentes probabilidades de recompensa para cada ação.

Os detalhes estão documentados abaixo:

### 0. Dataprep

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# ----------------------------------------------------
# Leitura dos dados
# ----------------------------------------------------

input_path = Path(
    "../data/processed/bank_marketing_processed.csv"
)

df = pd.read_csv(input_path)

print(f"Dataset carregado: {df.shape}")

# ----------------------------------------------------
# Separação do target e treino/teste
# ----------------------------------------------------

X = df.drop(columns=["y"])
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Treino: {X_train.shape}")
print(f"Teste : {X_test.shape}")

Dataset carregado: (41188, 25)


### 1. Definição dos braços (arms)

O problema foi estruturado considerando os canais de contato disponíveis no histórico:

* `cellular`
* `telephone`

Cada canal representa um **arm** que pode ser selecionado pelo algoritmo.

In [3]:
# Verifica canais disponíveis
print(df["contact"].value_counts())

contact
cellular     26144
telephone    15044
Name: count, dtype: int64


In [4]:
arms = {
    0: "cellular",
    1: "telephone"
}

n_arms = len(arms)

print(arms)

{0: 'cellular', 1: 'telephone'}


A recompensa (`reward`) foi definida a partir da variável `y`:

* `y = 1`: cliente converteu;
* `y = 0`: cliente não converteu.

Dessa forma, o algoritmo busca aprender a probabilidade de conversão associada a cada canal.

In [5]:
# Define recompensa 
X = df.drop(
    columns=["y", "contact"]
)

A = df["contact"]

R = df["y"]

# Carrega ação no split 
X_train, X_test, A_train, A_test, R_train, R_test = train_test_split(
    X,
    A,
    R,
    test_size=0.20,
    random_state=42,
    stratify=R
)

Agora cada observação do treino possui contexto (X), ação histórica (A) e recompensa (R).

### 2. Análise da distribuição histórica das ações

No conjunto de treinamento foram observadas as seguintes quantidades:

| Canal       | Observações | Taxa de conversão |
| ----------- | ----------: | ----------------: |
| `cellular`  |      20.908 |            14,70% |
| `telephone` |      12.042 |             5,30% |

Os dados históricos apresentam uma diferença significativa entre os canais. O canal `cellular` possui uma taxa de conversão aproximadamente três vezes maior que `telephone`.

Entretanto, essa diferença representa apenas uma **associação observada nos dados históricos** e não significa necessariamente que a escolha do canal seja a causa direta da maior conversão. Como os dados são observacionais, características dos clientes e a própria estratégia histórica de seleção do canal podem influenciar esse resultado.

In [6]:
print("\nDistribuição das ações no treino:")
print(A_train.value_counts())

print("\nTaxa de conversão por ação:")
print(
    pd.crosstab(
        A_train,
        R_train,
        normalize="index"
    )
)


Distribuição das ações no treino:
contact
cellular     20908
telephone    12042
Name: count, dtype: int64

Taxa de conversão por ação:
y                 0         1
contact                      
cellular   0.852975  0.147025
telephone  0.947019  0.052981


### 3. Inicialização dos parâmetros

Para cada arm foi utilizada uma distribuição Beta:

[theta_a sim Beta(alpha_a, beta_a)]

Inicialmente, os parâmetros foram definidos como: [alpha = 1] e [beta = 1]

Essa configuração representa uma distribuição uniforme e, portanto, não estabelece preferência inicial por nenhum dos canais.

A cada observação histórica:

* se `reward = 1`, o parâmetro `alpha` é incrementado;
* se `reward = 0`, o parâmetro `beta` é incrementado.

Assim:

[alpha_a = 1 + {número de conversões do arm}]

[beta_a = 1 + {número de não conversões do arm}]

Após o aprendizado offline, cada arm possui uma distribuição posterior que representa a incerteza sobre sua taxa de conversão.

In [8]:
# Parâmetros iniciais
alpha = {
    arm: 1
    for arm in arms.values()
}

beta = {
    arm: 1
    for arm in arms.values()
}


print("Parâmetros iniciais:")

for arm in arms.values():

    print(
        f"{arm}: "
        f"alpha={alpha[arm]}, "
        f"beta={beta[arm]}"
    )

Parâmetros iniciais:
cellular: alpha=1, beta=1
telephone: alpha=1, beta=1


In [9]:
# ATUALIZAÇÃO DOS POSTERIORES

for action, reward in zip(A_train, R_train):

    if reward == 1:

        alpha[action] += 1

    else:

        beta[action] += 1


print("\nParâmetros após aprendizado offline:")

for arm in arms.values():

    print(
        f"{arm}: "
        f"alpha={alpha[arm]}, "
        f"beta={beta[arm]}"
    )


Parâmetros após aprendizado offline:
cellular: alpha=3075, beta=17835
telephone: alpha=639, beta=11405


### 4. Seleção das ações

Para cada observação do conjunto de teste, foi realizada uma amostragem da distribuição posterior de cada arm.

O algoritmo seleciona o canal cuja amostra apresenta o maior valor:

[
a_t = \arg\max_a \theta_a
]

Essa estratégia permite equilibrar:

* **exploração:** testar ações cuja incerteza ainda pode permitir resultados melhores;
* **exploração do conhecimento atual:** favorecer ações que apresentam maior probabilidade de recompensa.

Como o `cellular` apresentou uma taxa histórica de conversão consideravelmente superior, espera-se que ele seja selecionado com maior frequência. Ainda assim, o Thompson Sampling pode ocasionalmente selecionar `telephone` devido à incerteza representada pelas distribuições posteriores.

In [13]:
import numpy as np
rng = np.random.default_rng(42)

n_simulations = len(A_test)

selected_actions = []

sampled_probabilities = []


for _ in range(n_simulations):

    samples = {}

    # ----------------------------------------
    # Amostra uma probabilidade para cada arm
    # ----------------------------------------

    for arm in arms.values():

        samples[arm] = rng.beta(
            alpha[arm],
            beta[arm]
        )

    # ----------------------------------------
    # Escolhe o arm com maior amostra
    # ----------------------------------------

    selected_arm = max(
        samples,
        key=samples.get
    )

    selected_actions.append(
        selected_arm
    )

    sampled_probabilities.append(
        samples
    )


selected_actions = np.array(
    selected_actions
)


print(
    "Distribuição das ações escolhidas:"
)

print(
    pd.Series(
        selected_actions
    ).value_counts()
)

Distribuição das ações escolhidas:
cellular    8238
Name: count, dtype: int64


### 5. Resultado da primeira simulação

A simulação foi realizada sobre **8.238 observações** do conjunto de teste.

| Métrica                             |  Resultado |
| ----------------------------------- | ---------: |
| Total de observações                |  **8.238** |
| Ações coincidentes                  |  **5.236** |
| Taxa de coincidência                | **63,56%** |
| Reward médio nas ações coincidentes | **14,88%** |

A **taxa de coincidência de 63,56%** indica que, em 5.236 das 8.238 observações, o canal escolhido pelo Thompson Sampling coincidiu com o canal que havia sido utilizado historicamente para aquele cliente.

O **reward médio de 14,88%** nas observações coincidentes representa a taxa de conversão observada entre os casos em que a ação escolhida pelo algoritmo foi igual à ação histórica.

In [14]:
A_test_array = A_test.to_numpy()
R_test_array = R_test.to_numpy()

# ------------------------------------------------------------
# Quando o Thompson escolheu a mesma ação histórica
# ------------------------------------------------------------

matched = (
    selected_actions
    == A_test_array
)


print(
    f"Total de observações: {len(A_test_array)}"
)

print(
    f"Ações coincidentes: {matched.sum()}"
)

print(
    f"Taxa de coincidência: {matched.mean():.4f}"
)


# ------------------------------------------------------------
# Reward observado nas ações coincidentes
# ------------------------------------------------------------

if matched.sum() > 0:

    observed_rewards = R_test_array[
        matched
    ]

    print(
        f"Reward médio nas ações coincidentes: "
        f"{observed_rewards.mean():.4f}"
    )

Total de observações: 8238
Ações coincidentes: 5236
Taxa de coincidência: 0.6356
Reward médio nas ações coincidentes: 0.1488


### 6. Limitação da avaliação

Esses resultados **não devem ser interpretados como a taxa real de conversão ou como a performance definitiva do Thompson Sampling**.

O principal motivo é que estamos trabalhando com **aprendizado offline a partir de dados observacionais**. Para cada cliente, conhecemos apenas o resultado da ação que realmente foi aplicada historicamente.

Por exemplo, se o histórico registra:

```text
Cliente → telephone → y = 0
```

não sabemos qual teria sido o resultado caso `cellular` tivesse sido utilizado.

Portanto, não é possível afirmar diretamente que:

```text
Thompson Sampling → cellular → y = 1
```

apenas porque o algoritmo escolheu `cellular`.

A taxa de coincidência utilizada nesta etapa é, portanto, uma **métrica exploratória para validar a implementação**, e não uma avaliação contrafactual completa.

### 7. Próxima etapa

Esta primeira implementação utiliza somente a informação agregada por ação:

[
P(reward \mid action)
]

Portanto, todos os clientes são tratados da mesma maneira.

A próxima etapa será incorporar o **contexto do cliente**, utilizando suas características para estimar diferentes probabilidades de recompensa para cada ação:

[
P(reward \mid contexto, action)
]

Isso permitirá que o algoritmo deixe de simplesmente aprender que `cellular` possui uma conversão histórica maior e passe a aprender **qual canal apresenta maior potencial para diferentes perfis de clientes**.

Essa evolução corresponde ao **Contextual Thompson Sampling**, que é a abordagem mais alinhada ao objetivo do projeto.


In [15]:
# Variáveis que não podem fazer parte do contexto
# ------------------------------------------------------------
# y       -> recompensa
# contact -> ação
# duration -> informação disponível somente após o contato

context_features = [
    col
    for col in X_train.columns
    if col not in [
        "contact",
        "duration"
    ]
]

print("Features utilizadas como contexto:")
print(context_features)

print(
    f"\nQuantidade de features de contexto: "
    f"{len(context_features)}"
)

Features utilizadas como contexto:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'previous_contact', 'previous_success', 'age_group', 'financial_risk', 'engagement_score']

Quantidade de features de contexto: 23


In [16]:
# IDENTIFICAÇÃO DOS TIPOS DE FEATURES
categorical_features = (
    X_train[context_features]
    .select_dtypes(include="object")
    .columns
    .tolist()
)

numeric_features = (
    X_train[context_features]
    .select_dtypes(exclude="object")
    .columns
    .tolist()
)

print("Features categóricas:")
print(categorical_features)

print("\nFeatures numéricas:")
print(numeric_features)

Features categóricas:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'month', 'day_of_week', 'poutcome', 'age_group']

Features numéricas:
['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'previous_contact', 'previous_success', 'financial_risk', 'engagement_score']


In [17]:
# TRANSFORMAÇÃO DO CONTEXTO

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer


preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)


X_train_context = preprocessor.fit_transform(
    X_train[context_features]
)

X_test_context = preprocessor.transform(
    X_test[context_features]
)


print(
    "Shape do contexto de treino:",
    X_train_context.shape
)

print(
    "Shape do contexto de teste:",
    X_test_context.shape
)

Shape do contexto de treino: (32950, 69)
Shape do contexto de teste: (8238, 69)


In [18]:
# INTERCEPTO
# Vamos adicionar uma coluna de 1s para permitir que o modelo aprenda um intercepto
X_train_context = np.column_stack(
    [
        np.ones(X_train_context.shape[0]),
        X_train_context
    ]
)

X_test_context = np.column_stack(
    [
        np.ones(X_test_context.shape[0]),
        X_test_context
    ]
)

print(
    "Shape final do contexto de treino:",
    X_train_context.shape
)

print(
    "Shape final do contexto de teste:",
    X_test_context.shape
)

Shape final do contexto de treino: (32950, 70)
Shape final do contexto de teste: (8238, 70)


In [20]:
# ============================================================
# ETAPA 15 — ESTRUTURA DO LINEAR THOMPSON SAMPLING
# ============================================================

import numpy as np


# ------------------------------------------------------------
# Número de dimensões do contexto
# ------------------------------------------------------------

n_features = X_train_context.shape[1]


# ------------------------------------------------------------
# Regularização
# ------------------------------------------------------------

lambda_reg = 1.0


# ------------------------------------------------------------
# Inicialização dos parâmetros por arm
# ------------------------------------------------------------

A = {}

b = {}

for arm in arms.values():

    A[arm] = (
        lambda_reg
        * np.eye(n_features)
    )

    b[arm] = np.zeros(
        n_features
    )


print(
    f"Número de dimensões do contexto: {n_features}"
)

for arm in arms.values():

    print(
        f"{arm}: "
        f"A={A[arm].shape}, "
        f"b={b[arm].shape}"
    )

Número de dimensões do contexto: 70
cellular: A=(70, 70), b=(70,)
telephone: A=(70, 70), b=(70,)


In [21]:
# ============================================================
# ETAPA 16 — APRENDIZADO OFFLINE
# ============================================================

for i in range(
    len(X_train_context)
):

    x = X_train_context[i]

    action = A_train.iloc[i]

    reward = R_train.iloc[i]

    # Atualização do braço correspondente
    A[action] += np.outer(
        x,
        x
    )

    b[action] += (
        reward * x
    )


print(
    "Aprendizado offline concluído."
)

Aprendizado offline concluído.


In [22]:
# ============================================================
# ETAPA 17 — POSTERIOR DOS BRAÇOS
# ============================================================

posterior_mean = {}

posterior_covariance = {}

for arm in arms.values():

    covariance = np.linalg.inv(
        A[arm]
    )

    mean = covariance @ b[arm]

    posterior_mean[arm] = mean

    posterior_covariance[arm] = covariance


for arm in arms.values():

    print(
        f"\nArm: {arm}"
    )

    print(
        "Norma do vetor de parâmetros:",
        np.linalg.norm(
            posterior_mean[arm]
        )
    )


Arm: cellular
Norma do vetor de parâmetros: 0.5915733412554809

Arm: telephone
Norma do vetor de parâmetros: 0.5923361711225555


In [23]:
# ============================================================
# ETAPA 18 — CONTEXTUAL THOMPSON SAMPLING
# ============================================================

rng = np.random.default_rng(42)

selected_actions = []

sampled_rewards = []


for x in X_test_context:

    arm_scores = {}

    for arm in arms.values():

        theta_sample = rng.multivariate_normal(
            mean=posterior_mean[arm],
            cov=posterior_covariance[arm]
        )

        predicted_reward = (
            x @ theta_sample
        )

        arm_scores[arm] = predicted_reward


    selected_arm = max(
        arm_scores,
        key=arm_scores.get
    )

    selected_actions.append(
        selected_arm
    )

    sampled_rewards.append(
        arm_scores
    )


selected_actions = np.array(
    selected_actions
)

In [24]:
# ============================================================
# ETAPA 19 — DISTRIBUIÇÃO DAS AÇÕES ESCOLHIDAS
# ============================================================

action_distribution = (
    pd.Series(
        selected_actions
    )
    .value_counts()
)

print(
    "Distribuição das ações escolhidas:"
)

print(
    action_distribution
)


print(
    "\nPercentual das ações escolhidas:"
)

print(
    (
        action_distribution
        / len(selected_actions)
    ).round(4)
)

Distribuição das ações escolhidas:
cellular     5889
telephone    2349
Name: count, dtype: int64

Percentual das ações escolhidas:
cellular     0.7149
telephone    0.2851
Name: count, dtype: float64


In [44]:
# ============================================================
# ETAPA 20 — REWARD ESTIMADO POR AÇÃO
# ============================================================

scores_df = pd.DataFrame(
    sampled_rewards
)

print(
    scores_df.describe()
)

          cellular    telephone
count  8238.000000  8238.000000
mean      0.162988     0.082832
std       0.163664     0.160383
min      -0.309690    -0.421766
25%       0.056330    -0.004170
50%       0.112574     0.047624
75%       0.231321     0.114109
max       1.160321     1.346247


In [26]:
print(
    "\nReward estimado médio por ação:"
)

print(
    scores_df.mean()
)


Reward estimado médio por ação:
cellular     0.162988
telephone    0.082832
dtype: float64


O Contextual Thompson Sampling foi treinado utilizando 32.950 observações e uma representação final de 70 dimensões por cliente. Foram mantidos parâmetros independentes para cada ação (cellular e telephone), permitindo estimar a recompensa condicionada ao contexto e à ação. 

O modelo utilizou o histórico de interações entre clientes, canais e conversões para aprender uma relação contextual entre as características dos clientes e a recompensa esperada de cada ação. Durante a simulação, o algoritmo selecionou cellular em 71,49% das observações e telephone em 28,51%. Para cada novo cliente, os scores dos diferentes canais são recalculados a partir de seu contexto e de uma amostra do posterior dos respectivos arms, permitindo que a ação seja selecionada individualmente. Quando uma nova recompensa é observada, os parâmetros do arm escolhido são atualizados, permitindo que decisões futuras sejam influenciadas pela nova informação. Na simulação realizada, os scores médios estimados foram 0,163 para cellular e 0,083 para telephone. Esses valores são estimativas do modelo e não representam taxas de conversão observadas.

* O score médio estimado para cellular foi 0,162988, enquanto para telephone foi 0,082832. Como o Thompson Sampling seleciona a ação com maior score amostrado, o cellular apresentou, nessa simulação, maior recompensa estimada que o telephone.
* O modelo considera cellular uma ação mais promissora que telephone em termos de reward estimado.

In [27]:
# ============================================================
# ANÁLISE 1 — DISTRIBUIÇÃO DAS AÇÕES POR PERFIL
# ============================================================

analysis_df = X_test.copy()

analysis_df["selected_action"] = selected_actions


features_to_analyze = [
    "age_group",
    "financial_risk",
    "previous_success",
    "previous_contact",
    "engagement_score",
    "job",
    "education",
    "marital",
    "month"
]


for feature in features_to_analyze:

    print("\n" + "=" * 70)
    print(f"FEATURE: {feature}")
    print("=" * 70)

    distribution = pd.crosstab(
        analysis_df[feature],
        analysis_df["selected_action"],
        normalize="index"
    ) * 100

    print(
        distribution.round(2)
    )


FEATURE: age_group
selected_action  cellular  telephone
age_group                           
adult               69.85      30.15
elderly             68.99      31.01
senior              72.03      27.97
young               71.88      28.12
young_adult         73.25      26.75

FEATURE: financial_risk
selected_action  cellular  telephone
financial_risk                      
0                   72.45      27.55
1                   70.77      29.23
2                   71.01      28.99

FEATURE: previous_success
selected_action   cellular  telephone
previous_success                     
0                    72.07      27.93
1                    54.10      45.90

FEATURE: previous_contact
selected_action   cellular  telephone
previous_contact                     
0                    74.18      25.82
1                    53.80      46.20

FEATURE: engagement_score
selected_action   cellular  telephone
engagement_score                     
1                    73.74      26.26
2           

| Feature                | Evidência observada                                         | Interpretação                                                                                                                                                    |
| ---------------------- | ----------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **`month`**            | `telephone` varia de **7,47% a 67,64%**                     | **Maior influência observada.** O modelo altera fortemente a escolha do canal conforme o mês e chega a inverter a preferência entre os canais.                   |
| **`previous_success`** | `telephone`: **27,93% → 45,90%**                            | **Forte influência.** O histórico de sucesso altera significativamente a escolha do canal.                                                                       |
| **`previous_contact`** | `telephone`: **25,82% → 46,20%**                            | **Forte influência.** O histórico de contato também modifica consideravelmente a política de escolha.                                                            |
| **`job`**              | `telephone`: **25,16% a 38,46%**                            | **Influência moderada.** Diferentes ocupações apresentam diferentes preferências estimadas de canal.                                                             |
| **`engagement_score`** | `telephone`: aproximadamente **20% a 67%** em vários grupos | **Possível influência relevante**, porém grupos com poucos registros podem gerar percentuais instáveis. Deve ser analisada considerando o tamanho de cada grupo. |
| **`age_group`**        | `telephone`: **26,75% a 31,01%**                            | **Baixa influência.** Existe alguma diferenciação, mas a variação é relativamente pequena.                                                                       |
| **`education`**        | `telephone`: **26,07% a 32,93%**                            | **Baixa influência** na escolha entre os dois canais.                                                                                                            |
| **`marital`**          | `telephone`: **13,33% a 30,20%**                            | **Baixa a moderada influência**, embora a categoria `unknown` possa ter poucos registros.                                                                        |
| **`financial_risk`**   | `telephone`: **27,55% a 29,23%**                            | **Baixa influência.** A distribuição praticamente não muda entre os níveis de risco.                                                                             |


Os resultados indicam que o modelo está apresentando comportamento contextual, em vez de simplesmente escolher o canal cellular de forma fixa. Embora a distribuição geral das ações tenha sido de aproximadamente 71,49% para cellular e 28,51% para telephone, a análise condicionada às características dos clientes mostrou que essa preferência varia de acordo com o contexto.

A variável month apresentou a maior diferenciação, com a escolha de telephone variando de 7,47% a 67,64%, chegando inclusive a superar cellular em alguns meses. As variáveis previous_success e previous_contact também apresentaram mudanças relevantes na distribuição das ações. job apresentou influência moderada, enquanto financial_risk, education e age_group produziram alterações menores.

Portanto, há evidências de que o Contextual Thompson Sampling está utilizando as características dos clientes para adaptar a escolha do canal, em vez de aplicar uma única estratégia para toda a população.

É importante destacar que essa análise não representa uma feature importance tradicional e não permite afirmar causalidade. Ela mostra como a política do Bandit distribui suas ações entre diferentes grupos de clientes. Para avaliar a importância das variáveis de forma mais rigorosa, o próximo passo recomendado é realizar um ablation study, removendo cada feature individualmente e comparando o comportamento e o desempenho do modelo.

### Avaliação

In [29]:
# ============================================================
# AÇÃO HISTÓRICA REAL
# ============================================================

action_test = df.loc[X_test.index, "contact"].reset_index(drop=True)

y_test_eval = df.loc[X_test.index, "y"].reset_index(drop=True)

print("Distribuição das ações históricas:")
print(action_test.value_counts())

print("\nTaxa de conversão por ação:")
print(
    pd.crosstab(
        action_test,
        y_test_eval,
        normalize="index"
    ).round(4)
)

Distribuição das ações históricas:
contact
cellular     5236
telephone    3002
Name: count, dtype: int64

Taxa de conversão por ação:
y               0       1
contact                  
cellular   0.8512  0.1488
telephone  0.9504  0.0496


In [31]:
# ============================================================
# 2. AÇÃO ESCOLHIDA PELO THOMPSON SAMPLING
# ============================================================

thompson_action = pd.Series(
    selected_actions
).reset_index(drop=True)

print("Distribuição das ações escolhidas pelo Thompson:")
print(thompson_action.value_counts())

print("\nPercentual das ações escolhidas:")
print(
    thompson_action
    .value_counts(normalize=True)
    .round(4)
)

Distribuição das ações escolhidas pelo Thompson:
cellular     5889
telephone    2349
Name: count, dtype: int64

Percentual das ações escolhidas:
cellular     0.7149
telephone    0.2851
Name: proportion, dtype: float64


In [32]:
# ============================================================
# 3. PREPARAÇÃO DO REWARD
# ============================================================

y_test_eval = y_test_eval.copy()

if y_test_eval.dtype == "object":
    y_test_eval = (
        y_test_eval
        .map({
            "yes": 1,
            "no": 0,
            "sim": 1,
            "não": 0
        })
    )

print("Distribuição do reward:")
print(y_test_eval.value_counts())

Distribuição do reward:
y
0    7310
1     928
Name: count, dtype: int64


In [33]:
# ============================================================
# 4. BASE DE AVALIAÇÃO
# ============================================================

evaluation = pd.DataFrame({
    "historical_action": action_test.values,
    "thompson_action": thompson_action.values,
    "reward": y_test_eval.values
})

print(evaluation.head())
print("\nShape:", evaluation.shape)

  historical_action thompson_action  reward
0          cellular       telephone       0
1          cellular        cellular       0
2          cellular        cellular       0
3         telephone        cellular       0
4          cellular       telephone       0

Shape: (8238, 3)


In [34]:
# ============================================================
# 5. REPLAY DO THOMPSON
# ============================================================

evaluation["thompson_match"] = (
    evaluation["thompson_action"]
    ==
    evaluation["historical_action"]
)

thompson_replay = evaluation[
    evaluation["thompson_match"]
].copy()

print(
    f"Total de observações: {len(evaluation)}"
)

print(
    f"Ações coincidentes: {len(thompson_replay)}"
)

print(
    f"Taxa de coincidência: "
    f"{evaluation['thompson_match'].mean():.4f}"
)

print(
    f"Reward médio nos matches: "
    f"{thompson_replay['reward'].mean():.4f}"
)

Total de observações: 8238
Ações coincidentes: 3831
Taxa de coincidência: 0.4650
Reward médio nos matches: 0.1441


In [35]:
# ============================================================
# 6. RECUPERAR BASE ORIGINAL PARA O BASELINE
# ============================================================

X_train_original = (
    df.loc[X_train.index]
    .drop(columns=["y"])
)

X_test_original = (
    df.loc[X_test.index]
    .drop(columns=["y"])
)

y_train_original = df.loc[
    X_train.index,
    "y"
]

print("X_train_original:", X_train_original.shape)
print("X_test_original :", X_test_original.shape)

X_train_original: (32950, 24)
X_test_original : (8238, 24)


In [36]:
# ============================================================
# 7. BASELINE — CANAL MAJORITÁRIO POR PERFIL
# ============================================================

perfil = [
    "job",
    "education",
    "marital",
    "month",
    "previous_contact",
    "previous_success",
    "age_group",
    "financial_risk",
    "engagement_score"
]

train_baseline = X_train_original.copy()

train_baseline["action"] = train_baseline["contact"]


# Ação majoritária global
acao_global = train_baseline["action"].mode()[0]


# Ação majoritária por perfil
baseline_por_perfil = (
    train_baseline
    .groupby(perfil)["action"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)


# Função de previsão
def prever_baseline(row):

    chave = tuple(
        row[col]
        for col in perfil
    )

    return baseline_por_perfil.get(
        chave,
        acao_global
    )


baseline_action = (
    X_test_original
    .apply(prever_baseline, axis=1)
    .reset_index(drop=True)
)

print("Distribuição das ações escolhidas pelo baseline:")

print(
    baseline_action.value_counts()
)

print("\nPercentual:")

print(
    baseline_action
    .value_counts(normalize=True)
    .round(4)
)

Distribuição das ações escolhidas pelo baseline:
cellular     6074
telephone    2164
Name: count, dtype: int64

Percentual:
cellular     0.7373
telephone    0.2627
Name: proportion, dtype: float64


In [37]:
# ============================================================
# 8. ADICIONAR BASELINE À AVALIAÇÃO
# ============================================================

evaluation["baseline_action"] = baseline_action.values

evaluation.head()

,historical_action,thompson_action,reward,thompson_match,baseline_action
0,cellular,telephone,0,False,cellular
1,cellular,cellular,0,True,cellular
2,cellular,cellular,0,True,cellular
3,telephone,cellular,0,False,cellular
4,cellular,telephone,0,False,cellular


In [38]:
# ============================================================
# 9. REPLAY DO BASELINE
# ============================================================

evaluation["baseline_match"] = (
    evaluation["baseline_action"]
    ==
    evaluation["historical_action"]
)

baseline_replay = evaluation[
    evaluation["baseline_match"]
].copy()

print(
    f"Total de observações: {len(evaluation)}"
)

print(
    f"Ações coincidentes: {len(baseline_replay)}"
)

print(
    f"Taxa de coincidência: "
    f"{evaluation['baseline_match'].mean():.4f}"
)

print(
    f"Reward médio nos matches: "
    f"{baseline_replay['reward'].mean():.4f}"
)

Total de observações: 8238
Ações coincidentes: 6240
Taxa de coincidência: 0.7575
Reward médio nos matches: 0.1212


In [39]:
# ============================================================
# 10. COMPARAÇÃO FINAL
# ============================================================

comparison = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Thompson Sampling"
    ],

    "Observações avaliadas": [
        len(baseline_replay),
        len(thompson_replay)
    ],

    "Taxa de coincidência": [
        evaluation["baseline_match"].mean(),
        evaluation["thompson_match"].mean()
    ],

    "Reward médio": [
        baseline_replay["reward"].mean(),
        thompson_replay["reward"].mean()
    ],

    "Conversões observadas": [
        baseline_replay["reward"].sum(),
        thompson_replay["reward"].sum()
    ]
})

comparison

,Modelo,Observações avaliadas,Taxa de coincidência,Reward médio,Conversões observadas
0,Baseline,6240,0.757465,0.121154,756
1,Thompson Sampling,3831,0.465040,0.144088,552


In [41]:
# ============================================================
# 11. GANHO DO THOMPSON SOBRE O BASELINE
# ============================================================

reward_baseline = baseline_replay["reward"].mean()

reward_thompson = thompson_replay["reward"].mean()

ganho_absoluto = (
    reward_thompson -
    reward_baseline
)

ganho_percentual = (
    (reward_thompson / reward_baseline) - 1
) * 100

print(f"Reward Baseline : {reward_baseline:.4f}")
print(f"Reward Thompson : {reward_thompson:.4f}")
print(f"Ganho absoluto  : {ganho_absoluto:.4f}")
print(f"Ganho relativo  : {ganho_percentual:.2f}%")

Reward Baseline : 0.1212
Reward Thompson : 0.1441
Ganho absoluto  : 0.0229
Ganho relativo  : 18.93%


O Thompson Sampling apresentou reward médio 18,93% superior ao baseline.
* Baseline: 12,12%
* Thompson: 14,41%
* ganho absoluto: 2,29 pontos percentuais
* ganho relativo: 18,93%

In [42]:
print("Total:", len(evaluation))

print("\nHistórico:")
print(evaluation["historical_action"].value_counts())

print("\nThompson:")
print(evaluation["thompson_action"].value_counts())

print("\nMatches:")
print(
    evaluation["thompson_match"].value_counts()
)

print("\nTaxa de coincidência:")
print(
    evaluation["thompson_match"].mean()
)

Total: 8238

Histórico:
historical_action
cellular     5236
telephone    3002
Name: count, dtype: int64

Thompson:
thompson_action
cellular     5889
telephone    2349
Name: count, dtype: int64

Matches:
thompson_match
False    4407
True     3831
Name: count, dtype: int64

Taxa de coincidência:
0.4650400582665696


In [43]:
print(
    pd.crosstab(
        evaluation["historical_action"],
        evaluation["thompson_action"],
        margins=True
    )
)

thompson_action    cellular  telephone   All
historical_action                           
cellular               3359       1877  5236
telephone              2530        472  3002
All                    5889       2349  8238


#### Resultado da avaliação offline

Na avaliação offline por replay, o Contextual Thompson Sampling apresentou reward médio de 14,41%, enquanto o baseline apresentou reward médio de 12,12%. Isso representa um ganho absoluto de 2,29 pontos percentuais e um ganho relativo de aproximadamente 18,93% em relação ao baseline.

O Thompson Sampling apresentou uma taxa de coincidência de 46,50%, enquanto o baseline apresentou 75,75%. A menor taxa de coincidência do Thompson é esperada em parte devido à sua capacidade de explorar uma política diferente daquela observada historicamente.

Os resultados sugerem que, apesar de recomendar o canal histórico com menor frequência, o Thompson Sampling apresentou maior reward médio nas observações avaliáveis, indicando potencial para selecionar canais de contato mais eficientes de acordo com o contexto dos clientes.

Entretanto, essa avaliação é offline e baseada em dados históricos. Como o reward da ação alternativa não é observado para cada cliente, não é possível afirmar causalmente que o Thompson Sampling produziria 18,93% mais conversões em produção. Uma validação online ou um conjunto de dados com informações de propensão das ações seria necessário para confirmar esse ganho.

In [53]:
# ============================================================
# FEATURES COMPLETAS
# ============================================================

features_completas = [
    "age",
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "month",
    "day_of_week",
    "campaign",
    "pdays",
    "previous",
    "poutcome",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
    "previous_contact",
    "previous_success",
    "age_group",
    "financial_risk",
    "engagement_score",
]


# ============================================================
# FEATURES COMPORTAMENTAIS
# ============================================================

features_comportamentais = [
    "campaign",
    "previous",
    "previous_contact",
    "previous_success",
    "engagement_score",
]


# ============================================================
# CONFIGURAÇÕES
# ============================================================

configuracoes = {
    "completo": features_completas,
    "perfil": features_perfil,
    "historico": features_historico,
    "comportamental": features_comportamentais,
    "temporal": features_temporais,
    "economico": features_economicas,
    "demografico": features_demograficas,
    "enxuto": features_enxutas,
}

In [54]:
resultados = []

for nome, features in configuracoes.items():

    resultado = avaliar_configuracao(
        nome=nome,
        features=features,
        random_state=RANDOM_STATE
    )

    resultados.append(resultado)


resultados_df = pd.DataFrame(resultados)

resultados_df = (
    resultados_df
    .sort_values(
        "reward_medio",
        ascending=False
    )
    .reset_index(drop=True)
)

display(resultados_df)


CONFIGURAÇÃO: completo
Features originais: 23
Dimensões do contexto: 70

CONFIGURAÇÃO: perfil
Features originais: 9
Dimensões do contexto: 44

CONFIGURAÇÃO: historico
Features originais: 6
Dimensões do contexto: 9

CONFIGURAÇÃO: comportamental
Features originais: 5
Dimensões do contexto: 6

CONFIGURAÇÃO: temporal
Features originais: 2
Dimensões do contexto: 16

CONFIGURAÇÃO: economico
Features originais: 5
Dimensões do contexto: 6

CONFIGURAÇÃO: demografico
Features originais: 5
Dimensões do contexto: 31

CONFIGURAÇÃO: enxuto
Features originais: 7
Dimensões do contexto: 35


,configuracao,features_originais,features_contexto,observacoes_teste,observacoes_avaliadas,taxa_coincidencia,reward_medio,conversoes,cellular_pct,telephone_pct
0,temporal,2,16,8238,3552,0.431173,0.158221,562,0.751639,0.248361
1,economico,5,6,8238,3879,0.470867,0.158030,613,0.798131,0.201869
2,enxuto,7,35,8238,3559,0.432022,0.150885,537,0.698106,0.301894
3,perfil,9,44,8238,3563,0.432508,0.146786,523,0.678927,0.321073
4,demografico,5,31,8238,5090,0.617868,0.143418,730,0.900461,0.099539
5,completo,23,70,8238,3898,0.473173,0.143407,559,0.678563,0.321437
6,comportamental,5,6,8238,4865,0.590556,0.138952,676,0.929959,0.070041
7,historico,6,9,8238,4843,0.587885,0.137725,667,0.927288,0.072712


| Modelo             |     Reward | Ganho vs baseline |
| ------------------ | ---------: | ----------------: |
| Baseline           |     12,12% |                 — |
| Thompson completo  |     14,34% |            +18,3% |
| Thompson enxuto    |     15,09% |            +24,5% |
| Thompson econômico |     15,80% |        **+30,4%** |
| Thompson temporal  | **15,82%** |        **+30,5%** |
